In [ ]:
from datasets import load_dataset
import random as rd
import numpy as np
import itertools
from collections import Counter
from tqdm import tqdm
from encoders import *

In [9]:
ds = load_dataset("LabHC/bias_in_bios")
print(ds["train"])

Dataset({
    features: ['hard_text', 'profession', 'gender'],
    num_rows: 257478
})


In [ ]:
def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    rd.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    rd.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
cv_sets = []

for seed in seeds:  
    cv_sets.append(
        cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(ds["train"]["hard_text"], ds["train"]["profession"])), 
            k=num_folds,
            seed=seed
        )
    )

In [ ]:
vocab_size = 32000
min_frequency = 2
tokenizer = train_tokenizer(ds["train"]["hard_text"], vocab_size=vocab_size, min_frequency=min_frequency)

tokenizer.save(f"results/BPE_tokenizer_vs_{vocab_size}_mf_{min_frequency}.json")

In [7]:
output = tokenizer.encode(ds["test"]["hard_text"][0])

print(output.tokens)  # you can work with the string tokens
print(output.ids)     # or with the 0-based integer ids

['▁He', '▁specializes', '▁in', '▁development', '▁economics,', '▁household', '▁economics,', '▁and', '▁personnel', '▁economics.', '▁In', '▁2003', '▁he', '▁received', '▁his', '▁Ph.D.', '▁in', '▁Economics', '▁from', '▁the', '▁London', '▁School', '▁of', '▁Economics.', '▁Previously,', '▁he', '▁has', '▁worked', '▁as', '▁an', '▁Assistant', '▁Professor', '▁of', '▁Economics', '▁at', '▁the', '▁University', '▁of', '▁Chicago,', '▁Graduate', '▁School', '▁of', '▁Business', '▁and', '▁also', ',', '▁as', '▁a', '▁consultant', '▁for', '▁the', '▁World', '▁Bank.', '▁Centre', '▁for', '▁Research', '▁and', '▁Analysis', '▁of', '▁Migration']
[1565, 2179, 1513, 2126, 7604, 11897, 7604, 1512, 16391, 10490, 1646, 3502, 1614, 1827, 1620, 2143, 1513, 3868, 1609, 1518, 3782, 1830, 1522, 10097, 5525, 1614, 1580, 1954, 1589, 1628, 3380, 2630, 1522, 3868, 1571, 1518, 1601, 1522, 5854, 3949, 1830, 1522, 2907, 1512, 1724, 34, 1589, 1500, 3914, 1567, 1518, 3271, 21818, 3482, 1567, 2125, 1512, 6132, 1522, 13629]


In [ ]:
tf_idf_encoder = TfIdfEncoder(tokenizer, ds["train"]["hard_text"])

In [17]:
tf_idf_vector = tf_idf_encoder(ds["train"]["hard_text"][0])
print(len(tf_idf_vector))
print(tf_idf_vector.sum())

32000
3.677758869312834
